In [1]:
import importlib, nenufar_ui
importlib.reload(nenufar_ui)
from nenufar_ui import load_and_select_sb
from nenufar_ui import run_step1_ui
from nenufar_ui import run_step2_zoom_ui
from nenufar_ui import run_step3_calib_ui
from nenufar_ui import run_step4_wsclean_ui
from nenufar_ui import run_step4_quicklook_ui
from nenufar_ui import run_step5a_iocorrect_solve_ui
from nenufar_ui import run_step5b_iocorrect_apply_ui
from nenufar_ui import run_step5c_centroid_ui

# Scan & SB Selection

## Purpose
Locate SUN and CAS_A events for a given date, align SBs, and generate a processing plan (JSON).

## What it does
- Find SUN_TRACKING event
- Find all CAS_A_TRACKING candidates (pre & post)
- Match SBs by frequency
- Allow user SB + calibrator selection
- Save selected SB list to JSON

## Output
`selected_sb_pair_list.json`

Contains:
- selected_sb
- sun_ms
- casa_pre_ms
- casa_post_ms
- casa_ms (if chosen)

⚠ No DP3 processing is done here.
This step only builds the processing plan.

In [2]:
import sys
sys.path.insert(0, "/data/jzhang/20240310TYPEII_work")

from nenufar_ui import load_and_select_sb

BASE = "/databf/nenufar-nri/LT11"
WORKROOT = "/data/jzhang/nenufar_workflows"

load_and_select_sb(BASE, WORKROOT)

Output()

# Step-1 — Prepare Calibrator & Sun MS

## Goal
Prepare clean working copies of:
- CasA (calibrator)
- Sun (target)

No gain solving is done here.

---

## 1A — CasA: AOFlagger + Average

DP3:
steps=[flag,averager]
flag.type=aoflagger
averager.timestep=1
averager.freqstep=1

Output:
CasA_SBxxx_prep.MS

Purpose:
- Remove RFI
- Standardize resolution

---

## 1B — CasA: Preflag bad baselines (in-place)

DP3:
steps=[flag]
flag.type=preflagger
flag.baseline='MR102NEN&&*;MR103NEN&&*'

Purpose:
- Remove unstable baselines
- Stabilize calibration

---

## 1C — Sun: Clear flags → new copy

DP3:
steps=[flag]
flag.type=preflagger
flag.mode=clear
flag.baseline='*&&*'

Output:
SUN_SBxxx_prep.MS

Purpose:
- Reset flags
- Protect raw data
- Prepare for calibration

---

## Result per SB
CasA_SBxxx_prep.MS
SUN_SBxxx_prep.MS
(log files)

Step-1 = data hygiene stage.
Safe to re-run.

In [ ]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step1_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"
run_step1_ui(plan, out_root="/data/jzhang/nenufar_workflows/step1_outputs_20240310")

## Step-2 — Zooming in (ROI Time Selection)

**Goal**  
Extract a time window (Region of Interest, ROI) from the prepared SUN MS produced in Step-1.

**Input**
- `SUN_{SB}_prep.MS` (from Step-1)
- Start time (e.g. `2024/03/10/12:00:00`)
- End time (e.g. `2024/03/10/12:20:00`)

**Operation (DP3)**
```bash
DP3 msin=SUN_{SB}_prep.MS \
     msin.starttime='YYYY/MM/DD/HH:MM:SS' \
     msin.endtime='YYYY/MM/DD/HH:MM:SS' \
     steps=[] \
     msout=SUN_{SB}_ROI.MS


**Output:**

SUN_{SB}_ROI.MS

**Located in:**

step2_outputs_YYYYMMDD/ROI/SBxxx/

In [ ]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step2_zoom_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"

run_step2_zoom_ui(
    plan_json_path=plan,
    step1_root="/data/jzhang/nenufar_workflows/step1_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step2_outputs_20240310",
    default_start="2024/03/10/12:08:00",
    default_end="2024/03/10/12:38:00",
)

## Step-3 — DI Calibration (CasA → Sun ROI)

### Goal
Use the **CasA calibrator** to derive per-antenna gain solutions, then **apply** them to the **Sun ROI MS**, producing a calibrated data column (**CORR_NO_BEAM**).

> Note: **applybeam is intentionally NOT used** for NenuFAR solar imaging (cannot apply beam in this workflow).  
> We stop at: **gaincal (predict/solve) + applycal**.

---

### Inputs
For each selected subband `SBxxx.MS`:

1. **Calibrator MS (CasA)**  
   From **Step-1 outputs** (selected by `dropdown/pre/post`):
   - `step1_root/SBxxx/CasA_SBxxx_prep.MS`

2. **Target MS (Sun ROI)**  
   From **Step-2 outputs**:
   - `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`

3. **Calibrator sky model database**
   - `CasA.sourcedb` (pre-generated, provided by you)

---

### What it runs (per SB)
#### 3A) `gaincal` on CasA (solve gains)
- Generates a parset:
  - `step3_root/SBxxx/SBxxx_gaincal.parset`
- Runs DP3 gaincal, producing a solution table:
  - `step3_root/SBxxx/CasA_SBxxx_prep.MS/instrument`

#### 3B) `applycal` on Sun ROI (apply gains)
- Generates a parset:
  - `step3_root/SBxxx/SBxxx_applycal.parset`
- Runs DP3 applycal on:
  - `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`
- Writes calibrated visibilities into a **new data column**:
  - **`CORR_NO_BEAM`**  
  (in-place inside the same Sun ROI MS)

---

### Outputs
Per `SBxxx` you will get:

**Files (in Step-3 folder)**
- `step3_root/SBxxx/SBxxx_gaincal.parset`
- `step3_root/SBxxx/SBxxx_applycal.parset`
- logs:
  - `01_gaincal.log`
  - `02_applycal.log`

**Data product (in Step-2 ROI MS)**
- `step2_root/ROI/SBxxx/SUN_SBxxx_ROI.MS`
  - now contains a new column: **`CORR_NO_BEAM`**

---

### Why this matters
After Step-3, the Sun ROI dataset is calibrated (direction-independent, no beam) and ready for imaging steps that expect:
- stable gain-corrected visibilities
- `CORR_NO_BEAM` as the calibrated column

---

### Quick checks
**1) confirm CORR_NO_BEAM exists**
```bash
taql "select COLNAME() from /path/to/SUN_SBxxx_ROI.MS"

In [ ]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step3_calib_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"

run_step3_calib_ui(
    plan_json_path=plan,
    step1_root="/data/jzhang/nenufar_workflows/step1_outputs_20240310",
    step2_root="/data/jzhang/nenufar_workflows/step2_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step3_outputs_20240310",
    sourcedb="/data/jzhang/nenufarIMG_workflow/CasA.sourcedb",
)

In [7]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step4_wsclean_ui

plan = "/data/jzhang/nenufar_workflows/20240310_101000_20240310_135000_SUN_TRACKING/selected_sb_pair_list.json"

run_step4_wsclean_ui(
    plan_json_path=plan,
    step2_root="/data/jzhang/nenufar_workflows/step2_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
)

In [ ]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step4_quicklook_ui

run_step4_quicklook_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    out_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310/quicklook",
    default_sb="SB359",
)

In [21]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step5a_iocorrect_solve_ui

run_step5a_iocorrect_solve_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    # out_root 可不填：默认 /data/jzhang/nenufar_workflows/step_iocorrect_outputs_YYYYMMDD
    default_sb="SB410",
)

In [ ]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step5b_iocorrect_apply_ui

run_step5b_iocorrect_apply_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    # out_root 不填就会默认 /data/jzhang/nenufar_workflows/step_iocorrect_outputs_YYYYMMDD
    default_sb="SB410",
)

In [24]:
import sys
sys.path.insert(0, "/data/jzhang/nenufarIMG_workflow")

from nenufar_ui import run_step5c_centroid_ui

run_step5c_centroid_ui(
    step4_root="/data/jzhang/nenufar_workflows/step4_outputs_20240310",
    # step5b_root 可不填：会自动找 /data/jzhang/nenufar_workflows/step_iocorrect_outputs_YYYYMMDD 中最新的那个
    # step5b_root="/data/jzhang/nenufar_workflows/step_iocorrect_outputs_20260225",
    default_sb="SB410",
)